In [ ]:
# ============================================================
# OpenQSim Phase 0A — Full sweep from scratch
# ============================================================
# HOW TO USE:
#   1. Start a new Kaggle session (P100 GPU recommended)
#   2. Add Kaggle Secrets: KAGGLE_USERNAME, KAGGLE_KEY,
#      NVIDIA_API_KEY  (optional: NVIDIA_API_KEY_1, GROQ_API_KEY)
#   3. Dataset: harshalekkala1/openqsim-phase0a (already created)
#   4. Run this single cell — sweeps all 840 combos from index 0
# ============================================================

import sys, subprocess, os, shutil, importlib.util, re, json
from pathlib import Path

# ── 1. Install dependencies ───────────────────────────────
print("[1/5] Installing dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "qiskit>=2.0,<3.0", "qiskit-aer>=0.17",
    "pynvml", "nvidia-ml-py", "pyyaml",
    "pandas", "kaggle>=2.2.2", "requests",
    "python-dotenv", "networkx",
])
print("      Done.")

# ── 2. Clone latest code ─────────────────────────────────
print("[2/5] Cloning repository...")
REPO_URL = "https://github.com/lekkalaharsha/opensim-ai"
REPO_DIR = "/kaggle/working/opensim-ai"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
print("      Done.")
sys.path.insert(0, REPO_DIR)

# ── 3. Load secrets ───────────────────────────────────────
print("[3/5] Loading secrets...")
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "NVIDIA_API_KEY",
                "NVIDIA_API_KEY_1", "GROQ_API_KEY"):
        try:
            _v = _s.get_secret(_k)
            if _v:
                os.environ[_k] = _v
                print(f"      [OK] {_k} loaded")
        except Exception:
            pass
except Exception as e:
    print(f"      WARN secrets: {e}")

if os.environ.get("NVIDIA_API_KEY_1"):
    os.environ["NVIDIA_API_KEY_COUNT"] = "2"
    print("      [OK] Dual NVIDIA key mode")

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "harshalekkala1")
KAGGLE_DATASET  = f"{KAGGLE_USERNAME}/openqsim-phase0a"
OUTPUT_DIR      = "/kaggle/working/benchmark_outputs"
SWEEP_CONFIG    = f"{REPO_DIR}/benchmark/sweep_config_0a.yaml"
print(f"      Target dataset: {KAGGLE_DATASET}")

# ── 4. Patch imports + load modules ──────────────────────
print("[4/5] Patching imports...")
KAGGLE_MOD_DIR = f"{REPO_DIR}/kaggle"

def _patch(path):
    txt = open(path).read()
    txt = re.sub(r'from kaggle\.(\w+) import', r'from \1 import', txt)
    txt = re.sub(r'import kaggle\.(\w+)', r'import \1', txt)
    open(path, "w").write(txt)

for py in Path(KAGGLE_MOD_DIR).glob("*.py"):
    _patch(str(py))
_runner_py = f"{REPO_DIR}/benchmark/runner.py"
if os.path.exists(_runner_py):
    _patch(_runner_py)

sys.path.insert(0, KAGGLE_MOD_DIR)

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_r = _load("kaggle_runner",    f"{KAGGLE_MOD_DIR}/runner.py")
_a = _load("kaggle_assembler", f"{KAGGLE_MOD_DIR}/dataset_assembler.py")
_c = _load("kaggle_client",    f"{KAGGLE_MOD_DIR}/api_client.py")
KaggleRunner     = _r.KaggleRunner
DatasetAssembler = _a.DatasetAssembler
KaggleAPIClient  = _c.KaggleAPIClient

from backend.environment import collect_environment_metadata
env = collect_environment_metadata()
print(f"      GPU: {env.gpu_name} ({env.gpu_memory_mb} MB)")
print(f"      Qiskit: {env.qiskit_version}")

if KaggleAPIClient(dataset=KAGGLE_DATASET).dataset_exists():
    print(f"      [OK] Dataset exists: {KAGGLE_DATASET}")
else:
    print(f"      WARN Dataset not found: {KAGGLE_DATASET}")
    print(f"           Create it at kaggle.com/datasets/add — will zip as fallback")
print("      Modules loaded.")

# ── 5. Run sweep (840 combos from index 0) ───────────────
print("[5/5] Running benchmark sweep...")
runner = KaggleRunner(
    sweep_config_path=SWEEP_CONFIG,
    output_dir=OUTPUT_DIR,
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset=KAGGLE_DATASET,
    use_advisor=True,
)
sweep_result = runner.run()
print(f"\n      Sweep done: {sweep_result.completed_count}/{sweep_result.total_combinations}")
print(f"      OOM: {sweep_result.oom_count}  Errors: {sweep_result.error_count}")
print(f"      Time: {sweep_result.duration_seconds/3600:.1f}h")

# ── 6. Assemble dataset ───────────────────────────────────
print("\nAssembling dataset...")
assembler = DatasetAssembler(
    raw_dir=f"{OUTPUT_DIR}/raw",
    dataset_dir=f"{OUTPUT_DIR}/datasets/openqsim_v0.1-small",
)
manifest = assembler.assemble()
print(f"      {manifest.records} records | {manifest.successful_runs} successful")
print(f"      Backends: {manifest.backends} | Qubits: {manifest.qubit_range}")

# ── 7. Push to Kaggle (zip fallback on 403/404) ──────────
print("\nPushing dataset...")
dataset_dir = Path(OUTPUT_DIR)
(dataset_dir / "dataset-metadata.json").write_text(json.dumps({
    "title": "OpenQSim Phase 0A Benchmark Outputs",
    "id": KAGGLE_DATASET,
    "licenses": [{"name": "MIT"}],
}))
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(dataset_dir),
     "-m", f"Phase 0A complete: {manifest.records} records"],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print(f"[OK] Pushed to {KAGGLE_DATASET}")
else:
    zip_path = shutil.make_archive(
        "/kaggle/working/benchmark_outputs_backup", "zip", str(OUTPUT_DIR)
    )
    print(f"WARN Push failed: {r.stderr[:200]}")
    print(f"[OK] Zip saved: {zip_path} — download from Output tab")

print("\n" + "=" * 50)
print("OPENQSIM PHASE 0A SWEEP COMPLETE")
print("=" * 50)
print(f"  Records:   {manifest.records}")
print(f"  Successes: {manifest.successful_runs}")
print(f"  Backends:  {manifest.backends}")
print(f"  Qubits:    {manifest.qubit_range}")
print(f"  Depths:    {manifest.depth_range}")
print(f"  Dataset:   {KAGGLE_DATASET}")
print("=" * 50)
